## Graph Algorithms
Dijkstra's, A*, Bellman-Ford and Dynamic Programming on the Allen Mouse Brain Connectivity Graph.
Each notebook is self contained and reloads all data independently.

## Reload Allen Data and reconstruct connectivity matrix

In [1]:
import sqlite3
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from allensdk.core.mouse_connectivity_cache import MouseConnectivityCache
from scipy.spatial.distance import cdist

# reload everything from Phase 1
# (each notebook is independent - always reload at the top)
mcc = MouseConnectivityCache(manifest_file='../allen_cache/manifest.json')
structure_tree = mcc.get_structure_tree()
experiments = mcc.get_experiments(dataframe=True)

acronym_to_id = structure_tree.get_id_acronym_map()
id_to_acronym = {v: k for k, v in acronym_to_id.items()}
id_to_name    = structure_tree.get_name_map()

top_regions = (
    experiments['structure_abbrev']
    .value_counts()
    .head(20)
    .index.tolist()
)

region_coords = {}
for region in top_regions:
    region_exps = experiments[experiments['structure_abbrev'] == region]
    if len(region_exps) > 0:
        region_coords[region] = [
            region_exps['injection_x'].mean(),
            region_exps['injection_y'].mean(),
            region_exps['injection_z'].mean()
        ]

valid_regions  = [r for r in top_regions if r in region_coords]
coords_array   = np.array([region_coords[r] for r in valid_regions])
dist_matrix    = cdist(coords_array, coords_array, metric='euclidean')
epsilon        = 1e-6
strength_matrix = 1 / (dist_matrix + epsilon)
np.fill_diagonal(strength_matrix, 0)
strength_matrix /= strength_matrix.max()

print(f"Data reloaded: {len(valid_regions)} regions")

Data reloaded: 20 regions


## Build Graph with Cost Edges for Dijkstra's
Build the directed graph with dual edge attributes: weight (0-1 connection strength) and cost(1 - weight). Dijkstra's minimizes cost, which is equivalent to maximizing signal strength. Strong connections are cheap to traverse while weak connections are expensive.

In [2]:
G_dijkstra = nx.DiGraph()

for region in valid_regions:
    G_dijkstra.add_node(region, coords=region_coords[region])

THRESHOLD = 0.15
for i, source in enumerate(valid_regions):
    for j, target in enumerate(valid_regions):
        if i != j:
            weight = strength_matrix[i][j]
            if weight > THRESHOLD:
                # cost = 1 - weight
                # strong connection (0.95) → low cost (0.05) → Dijkstra prefers it
                # weak connection (0.15)   → high cost (0.85) → Dijkstra avoids it
                cost = 1 - weight
                G_dijkstra.add_edge(source, target,
                                    weight=weight,  # original strength
                                    cost=cost)      # what Dijkstra uses

print(f"Graph built: {G_dijkstra.number_of_nodes()} nodes, {G_dijkstra.number_of_edges()} edges")

Graph built: 20 nodes, 134 edges


## Run Dijkstra's Algorithm
Implement Dijkstra's algorithm to find the strongest signal pathway between brain regions. The cost is 1 - weight so Dijkstra naturally prefers strong connections. Three pathways queried representing key neuroscience circuits: perforant path, corticostriatal pathway and visual memory encoding route.

In [3]:
def find_neural_pathway(graph, source, target):
    try:
        path = nx.dijkstra_path(graph, source, target, weight='cost')

        # collect individual hop strengths
        hop_strengths = []
        print(f"\nStrongest pathway: {source} → {target}")
        print(f"{'─'*40}")
        print(f"Route: {' → '.join(path)}")
        print(f"Path length: {len(path)-1} hops")
        print(f"\nStep-by-step:")
        for i in range(len(path)-1):
            edge_data = graph[path[i]][path[i+1]]
            strength = edge_data['weight']
            hop_strengths.append(strength)
            print(f"  {path[i]} → {path[i+1]}: strength {strength:.4f}")

        # average strength is meaningful across any path length
        avg_strength = np.mean(hop_strengths)
        min_strength = np.min(hop_strengths)
        print(f"\nAverage hop strength: {avg_strength:.4f}")
        print(f"Weakest link:         {min_strength:.4f}")
        print(f"(weakest link = bottleneck of the pathway)")

        return path, avg_strength

    except nx.NetworkXNoPath:
        print(f"No path found between {source} and {target}")
        return None, None

# run the same three neuroscience queries
path1, str1 = find_neural_pathway(G_dijkstra, 'ENTl', 'DG')
path2, str2 = find_neural_pathway(G_dijkstra, 'MOs',  'CP')
path3, str3 = find_neural_pathway(G_dijkstra, 'VISp', 'CA1')


Strongest pathway: ENTl → DG
────────────────────────────────────────
Route: ENTl → DG
Path length: 1 hops

Step-by-step:
  ENTl → DG: strength 0.1650

Average hop strength: 0.1650
Weakest link:         0.1650
(weakest link = bottleneck of the pathway)

Strongest pathway: MOs → CP
────────────────────────────────────────
Route: MOs → ACAv → CP
Path length: 2 hops

Step-by-step:
  MOs → ACAv: strength 0.3949
  ACAv → CP: strength 0.1661

Average hop strength: 0.2805
Weakest link:         0.1661
(weakest link = bottleneck of the pathway)

Strongest pathway: VISp → CA1
────────────────────────────────────────
Route: VISp → DG → CA1
Path length: 2 hops

Step-by-step:
  VISp → DG: strength 0.1748
  DG → CA1: strength 0.5430

Average hop strength: 0.3589
Weakest link:         0.1748
(weakest link = bottleneck of the pathway)


## Save Dijkstra Results to DB
Save Dijkstra pathfinding results to the SQLite DB. The pathfinding results table stores algorithm name, source, target, full path string, hop count, and average strength. DROP and recreate ensures clean data on each run.

In [4]:
conn = sqlite3.connect('../db/neuro_nav.db')
cursor = conn.cursor()

# drop old table with bad values and recreate
cursor.execute('DROP TABLE IF EXISTS pathfinding_results')
cursor.execute('''
    CREATE TABLE IF NOT EXISTS pathfinding_results (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        algorithm TEXT NOT NULL,
        source TEXT NOT NULL,
        target TEXT NOT NULL,
        path TEXT NOT NULL,
        num_hops INTEGER,
        avg_pathway_strength REAL
    )
''')

paths_to_save = [
    ('dijkstra', 'ENTl', 'DG',   path1, str1),
    ('dijkstra', 'MOs',  'CP',   path2, str2),
    ('dijkstra', 'VISp', 'CA1',  path3, str3),
]

for algo, src, tgt, path, strength in paths_to_save:
    if path:
        cursor.execute('''
            INSERT INTO pathfinding_results
            (algorithm, source, target, path, num_hops, avg_pathway_strength)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (algo, src, tgt,
              ' → '.join(path),
              len(path)-1,
              round(strength, 4)))

conn.commit()
cursor.execute('SELECT * FROM pathfinding_results')
print("Saved pathfinding results:")
for row in cursor.fetchall():
    print(f"  {row}")

conn.close()

Saved pathfinding results:
  (1, 'dijkstra', 'ENTl', 'DG', 'ENTl → DG', 1, 0.165)
  (2, 'dijkstra', 'MOs', 'CP', 'MOs → ACAv → CP', 2, 0.2805)
  (3, 'dijkstra', 'VISp', 'CA1', 'VISp → DG → CA1', 2, 0.3589)


## A* Algorithm
Implement A* search using 3D spatial coordinates as the heuristic. A* estimates remaining cost using Elucidean distance to the target, guiding search spatially rather than exploring in all directions. Identical results to Dijkstra's confirm spatial embedding. The brain's stringest connections are also its most spatially direct.

In [5]:
def heuristic(region_a, region_b):
    '''
    estimates the cost remaining from region_a to region_b
    using 3D Euclidean distance between injection coordinates
    normalized to 0-1 range so it's comparable to our edge costs
    '''
    coords_a = np.array(region_coords[region_a])
    coords_b = np.array(region_coords[region_b])
    distance = np.linalg.norm(coords_a - coords_b)

    # normalize by max possible distance in our dataset
    max_dist = np.max(dist_matrix)
    return distance / max_dist

def find_neural_pathway_astar(graph, source, target):
    try:
        # nx.astar_path takes a heuristic function
        # it calls heuristic(current_node, target) at each step
        path = nx.astar_path(
            graph,
            source,
            target,
            heuristic=heuristic,
            weight='cost'
        )

        hop_strengths = []
        print(f"\nA* pathway: {source} → {target}")
        print(f"{'─'*40}")
        print(f"Route: {' → '.join(path)}")
        print(f"Path length: {len(path)-1} hops")
        print(f"\nStep-by-step:")
        for i in range(len(path)-1):
            edge_data = graph[path[i]][path[i+1]]
            strength = edge_data['weight']
            hop_strengths.append(strength)
            print(f"  {path[i]} → {path[i+1]}: strength {strength:.4f}")

        avg_strength = np.mean(hop_strengths)
        print(f"\nAverage hop strength: {avg_strength:.4f}")
        print(f"Weakest link:         {np.min(hop_strengths):.4f}")

        return path, avg_strength

    except nx.NetworkXNoPath:
        print(f"No path found between {source} and {target}")
        return None, None

# run the same three queries so we can compare with Dijkstra's
print("=" * 50)
print("A* RESULTS")
print("=" * 50)
path1_as, str1_as = find_neural_pathway_astar(G_dijkstra, 'ENTl', 'DG')
path2_as, str2_as = find_neural_pathway_astar(G_dijkstra, 'MOs',  'CP')
path3_as, str3_as = find_neural_pathway_astar(G_dijkstra, 'VISp', 'CA1')

# compare side by side
print("\n" + "=" * 50)
print("DIJKSTRA vs A* COMPARISON")
print("=" * 50)
queries = [
    ('ENTl→DG',  path1,    str1,    path1_as, str1_as),
    ('MOs→CP',   path2,    str2,    path2_as, str2_as),
    ('VISp→CA1', path3,    str3,    path3_as, str3_as),
]
for name, dp, ds, ap, as_ in queries:
    d_route = ' → '.join(dp) if dp else 'none'
    a_route = ' → '.join(ap) if ap else 'none'
    same = "✓ same" if dp == ap else "✗ different"
    print(f"\n{name}:")
    print(f"  Dijkstra: {d_route} (avg {ds:.4f})")
    print(f"  A*:       {a_route} (avg {as_:.4f})")
    print(f"  {same}")

A* RESULTS

A* pathway: ENTl → DG
────────────────────────────────────────
Route: ENTl → DG
Path length: 1 hops

Step-by-step:
  ENTl → DG: strength 0.1650

Average hop strength: 0.1650
Weakest link:         0.1650

A* pathway: MOs → CP
────────────────────────────────────────
Route: MOs → ACAv → CP
Path length: 2 hops

Step-by-step:
  MOs → ACAv: strength 0.3949
  ACAv → CP: strength 0.1661

Average hop strength: 0.2805
Weakest link:         0.1661

A* pathway: VISp → CA1
────────────────────────────────────────
Route: VISp → DG → CA1
Path length: 2 hops

Step-by-step:
  VISp → DG: strength 0.1748
  DG → CA1: strength 0.5430

Average hop strength: 0.3589
Weakest link:         0.1748

DIJKSTRA vs A* COMPARISON

ENTl→DG:
  Dijkstra: ENTl → DG (avg 0.1650)
  A*:       ENTl → DG (avg 0.1650)
  ✓ same

MOs→CP:
  Dijkstra: MOs → ACAv → CP (avg 0.2805)
  A*:       MOs → ACAv → CP (avg 0.2805)
  ✓ same

VISp→CA1:
  Dijkstra: VISp → DG → CA1 (avg 0.3589)
  A*:       VISp → DG → CA1 (avg 0.3589

## Save A* Results to DB
Append A* results to the pathfinding_results table alongside Dijkstra results. Both algorithms share the same table with an algorithm column, enabling direct SQL comparison. The verification query confirms both sets of results are stored correctly.

In [6]:
conn = sqlite3.connect('../db/neuro_nav.db')
cursor = conn.cursor()

astar_paths = [
    ('astar', 'ENTl', 'DG',  path1_as, str1_as),
    ('astar', 'MOs',  'CP',  path2_as, str2_as),
    ('astar', 'VISp', 'CA1', path3_as, str3_as),
]

for algo, src, tgt, path, strength in astar_paths:
    if path:
        cursor.execute('''
            INSERT INTO pathfinding_results
            (algorithm, source, target, path, num_hops, avg_pathway_strength)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (algo, src, tgt,
              ' → '.join(path),
              len(path)-1,
              round(strength, 4)))

conn.commit()
cursor.execute('''
    SELECT algorithm, source, target, path, avg_pathway_strength
    FROM pathfinding_results
    ORDER BY algorithm, source
''')
print("All pathfinding results in database:")
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}→{row[2]}: {row[3]} (avg {row[4]})")

conn.close()

All pathfinding results in database:
  [astar] ENTl→DG: ENTl → DG (avg 0.165)
  [astar] MOs→CP: MOs → ACAv → CP (avg 0.2805)
  [astar] VISp→CA1: VISp → DG → CA1 (avg 0.3589)
  [dijkstra] ENTl→DG: ENTl → DG (avg 0.165)
  [dijkstra] MOs→CP: MOs → ACAv → CP (avg 0.2805)
  [dijkstra] VISp→CA1: VISp → DG → CA1 (avg 0.3589)


## Bellman-Ford Neurodegeneration Simulation
Bellman-Ford neurodegeneration simulations. unlike Dijkstra's, Bellman-Ford handles negative edge weights. This is necessary when connection penalties push costs above 1.0. Three scenarios model Alzheimer;'s progression: healthy (0.193), early degeneration (0.113), and severe degeneration (0.053). A 73% signal loss mirrors clinical memory impairment.

In [7]:
# Bellman-Ford handles negative edge weights, which Dijkstra's cannot
# We use this to model neurodegeneration:
# when a connection weakens (like in Alzheimer's),
# we represent it as a negative weight penalty
# Bellman-Ford finds the best path even through degraded connections

def build_degraded_graph(base_graph, degraded_connections):
    '''
    takes our normal graph and weakens specific connections
    degraded_connections = list of (source, target, penalty) tuples
    penalty is subtracted from the edge weight to simulate degeneration
    '''
    G_degraded = base_graph.copy()

    for source, target, penalty in degraded_connections:
        if G_degraded.has_edge(source, target):
            old_weight = G_degraded[source][target]['weight']
            new_weight = old_weight - penalty
            # cost becomes negative when connection is severely degraded
            new_cost   = 1 - new_weight
            G_degraded[source][target]['weight'] = new_weight
            G_degraded[source][target]['cost']   = new_cost
            print(f"Degraded {source}→{target}: "
                  f"{old_weight:.4f} → {new_weight:.4f} "
                  f"(penalty: {penalty})")

    return G_degraded

def find_pathway_bellman_ford(graph, source, target):
    '''
    Bellman-Ford finds shortest path even with negative weights
    useful when degraded connections create negative costs
    '''
    try:
        # bellman_ford returns predecessor map and distance dict
        length, path = nx.single_source_bellman_ford(
            graph, source, target, weight='cost'
        )

        hop_strengths = []
        print(f"\nBellman-Ford pathway: {source} → {target}")
        print(f"{'─'*40}")
        print(f"Route: {' → '.join(path)}")
        print(f"Path length: {len(path)-1} hops")
        print(f"\nStep-by-step:")
        for i in range(len(path)-1):
            edge_data = graph[path[i]][path[i+1]]
            strength  = edge_data['weight']
            hop_strengths.append(strength)
            status = " ⚠ DEGRADED" if strength < 0.1 else ""
            print(f"  {path[i]} → {path[i+1]}: "
                  f"strength {strength:.4f}{status}")

        avg_strength = np.mean(hop_strengths)
        print(f"\nAverage hop strength: {avg_strength:.4f}")
        if avg_strength < 0:
            print("  ⚠ WARNING: pathway severely compromised")
        elif avg_strength < 0.15:
            print("  ⚠ WARNING: pathway weakly connected")
        else:
            print("  ✓ pathway intact")

        return path, avg_strength

    except nx.NetworkXNoPath:
        print(f"No path found between {source} and {target}")
        return None, None
    except nx.exception.NetworkXUnbounded:
        print(f"Negative cycle detected - graph is unstable")
        return None, None

# --- scenario: early alzheimer's ---
# the perforant path (ENTl → DG) and entorhinal connections
# are among the first casualties of Alzheimer's disease
# let's simulate progressive degeneration and see how
# the brain reroutes signals

print("=" * 50)
print("SCENARIO: HEALTHY BRAIN")
print("=" * 50)
path_healthy, str_healthy = find_pathway_bellman_ford(
    G_dijkstra, 'ENTl', 'CA1'
)

print("\n" + "=" * 50)
print("SCENARIO: EARLY NEURODEGENERATION")
print("simulating 50% degradation of ENTl connections")
print("=" * 50)
degraded_early = build_degraded_graph(G_dijkstra, [
    ('ENTl', 'DG',  0.08),   # perforant path weakened
    ('ENTl', 'CA1', 0.08),   # direct ENTl→CA1 weakened
])
path_early, str_early = find_pathway_bellman_ford(
    degraded_early, 'ENTl', 'CA1'
)

print("\n" + "=" * 50)
print("SCENARIO: SEVERE NEURODEGENERATION")
print("simulating 90% degradation of ENTl connections")
print("=" * 50)
degraded_severe = build_degraded_graph(G_dijkstra, [
    ('ENTl', 'DG',  0.14),
    ('ENTl', 'CA1', 0.14),
    ('ENTl', 'ENTm', 0.10),
])
path_severe, str_severe = find_pathway_bellman_ford(
    degraded_severe, 'ENTl', 'CA1'
)

# summary
print("\n" + "=" * 50)
print("DEGENERATION SUMMARY")
print("=" * 50)
scenarios = [
    ("Healthy",  path_healthy, str_healthy),
    ("Early",    path_early,   str_early),
    ("Severe",   path_severe,  str_severe),
]
for name, path, strength in scenarios:
    if path:
        route = ' → '.join(path)
        print(f"{name:8}: {route}")
        print(f"          avg strength: {strength:.4f}")

SCENARIO: HEALTHY BRAIN

Bellman-Ford pathway: ENTl → CA1
────────────────────────────────────────
Route: ENTl → CA1
Path length: 1 hops

Step-by-step:
  ENTl → CA1: strength 0.1927

Average hop strength: 0.1927
  ✓ pathway intact

SCENARIO: EARLY NEURODEGENERATION
simulating 50% degradation of ENTl connections
Degraded ENTl→DG: 0.1650 → 0.0850 (penalty: 0.08)
Degraded ENTl→CA1: 0.1927 → 0.1127 (penalty: 0.08)

Bellman-Ford pathway: ENTl → CA1
────────────────────────────────────────
Route: ENTl → CA1
Path length: 1 hops

Step-by-step:
  ENTl → CA1: strength 0.1127

Average hop strength: 0.1127
  ⚠ WARNING: pathway weakly connected

SCENARIO: SEVERE NEURODEGENERATION
simulating 90% degradation of ENTl connections
Degraded ENTl→DG: 0.1650 → 0.0250 (penalty: 0.14)
Degraded ENTl→CA1: 0.1927 → 0.0527 (penalty: 0.14)
Degraded ENTl→ENTm: 0.4795 → 0.3795 (penalty: 0.1)

Bellman-Ford pathway: ENTl → CA1
────────────────────────────────────────
Route: ENTl → CA1
Path length: 1 hops

Step-by-ste

## Save Bellman-Ford Results to DB
Save the Bellman-Ford Scenarios to the pathfinidng_results table. Each scenario is stored as a separate algorithm label, engabling SQL queries that track signal degradation across healthy, early and sever stages of  neurodegeneration. The full table now contains ersults from all three pathfinding algorithms.

In [8]:
conn = sqlite3.connect('../db/neuro_nav.db')
cursor = conn.cursor()

bf_paths = [
    ('bellman_ford_healthy', 'ENTl', 'CA1', path_healthy, str_healthy),
    ('bellman_ford_early',   'ENTl', 'CA1', path_early,   str_early),
    ('bellman_ford_severe',  'ENTl', 'CA1', path_severe,  str_severe),
]

for algo, src, tgt, path, strength in bf_paths:
    if path:
        cursor.execute('''
            INSERT INTO pathfinding_results
            (algorithm, source, target, path, num_hops, avg_pathway_strength)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', (algo, src, tgt,
              ' → '.join(path),
              len(path)-1,
              round(strength, 4) if strength else None))

conn.commit()
print("Bellman-Ford results saved.")
print("\nFull pathfinding table:")
cursor.execute('''
    SELECT algorithm, source, target, path, avg_pathway_strength
    FROM pathfinding_results
    ORDER BY algorithm
''')
for row in cursor.fetchall():
    print(f"  [{row[0]}] {row[1]}→{row[2]}: {row[3]} (avg {row[4]})")

conn.close()

Bellman-Ford results saved.

Full pathfinding table:
  [astar] ENTl→DG: ENTl → DG (avg 0.165)
  [astar] MOs→CP: MOs → ACAv → CP (avg 0.2805)
  [astar] VISp→CA1: VISp → DG → CA1 (avg 0.3589)
  [bellman_ford_early] ENTl→CA1: ENTl → CA1 (avg 0.1127)
  [bellman_ford_healthy] ENTl→CA1: ENTl → CA1 (avg 0.1927)
  [bellman_ford_severe] ENTl→CA1: ENTl → CA1 (avg 0.0527)
  [dijkstra] ENTl→DG: ENTl → DG (avg 0.165)
  [dijkstra] MOs→CP: MOs → ACAv → CP (avg 0.2805)
  [dijkstra] VISp→CA1: VISp → DG → CA1 (avg 0.3589)


## Sequence Alignment on Brain Connectivity Patterns
Needleman-Wunsch global sequence alignment adapted for brain connectivity. Each region's connectivity pattern (its row in the strength matrix) becomes a "sequence" of connection strengths. Alignment score measures wiring similarity. Regions with similar connectivity fingerprints likely serve similar functional roles. Directly analogous to DNA sequence homology analysis in bioinformatics.

In [9]:
# Each brain region has a "connectivity signature" -
# a vector of its connection strengths to all other regions
# We align these signatures like DNA sequences to find
# which regions have similar wiring patterns

def get_connectivity_signature(region, regions, matrix_df):
    '''
    returns the connectivity pattern of a region
    as a list of (target, strength) pairs, sorted by target name
    this is our "sequence" for alignment
    '''
    row = matrix_df.loc[region]
    # exclude self-connection
    signature = [(target, row[target])
                 for target in regions if target != region]
    signature.sort(key=lambda x: x[0])  # sort alphabetically for consistency
    return signature

def needleman_wunsch(sig_a, sig_b, gap_penalty=-0.1):
    '''
    Needleman-Wunsch global sequence alignment
    adapted for connectivity signatures instead of DNA bases

    instead of matching A/T/G/C, we match connection strengths
    score = 1 - |strength_a - strength_b|  (similar strengths = high score)
    gap penalty = cost of inserting a gap in the alignment

    returns: alignment score, aligned sequence a, aligned sequence b
    '''
    n = len(sig_a)
    m = len(sig_b)

    # --- initialize DP matrix ---
    # dp[i][j] = best alignment score for sig_a[:i] vs sig_b[:j]
    dp = np.zeros((n+1, m+1))

    # fill first row and column with gap penalties
    # (aligning against nothing costs gap_penalty per position)
    for i in range(n+1):
        dp[i][0] = i * gap_penalty
    for j in range(m+1):
        dp[0][j] = j * gap_penalty

    # --- fill DP matrix (the core recursion) ---
    for i in range(1, n+1):
        for j in range(1, m+1):
            # score for matching position i with position j
            # similar connection strengths = high match score
            strength_diff = abs(sig_a[i-1][1] - sig_b[j-1][1])
            match_score = 1 - strength_diff  # 1.0 = perfect match, 0.0 = very different

            # three choices at each cell (classic DP recursion):
            diagonal = dp[i-1][j-1] + match_score  # match/mismatch
            up        = dp[i-1][j]   + gap_penalty  # gap in sig_b
            left      = dp[i][j-1]   + gap_penalty  # gap in sig_a

            dp[i][j] = max(diagonal, up, left)

    # --- traceback to find the alignment ---
    aligned_a = []
    aligned_b = []
    i, j = n, m

    while i > 0 or j > 0:
        if i > 0 and j > 0:
            strength_diff = abs(sig_a[i-1][1] - sig_b[j-1][1])
            match_score = 1 - strength_diff
            if dp[i][j] == dp[i-1][j-1] + match_score:
                aligned_a.append(sig_a[i-1])
                aligned_b.append(sig_b[j-1])
                i -= 1
                j -= 1
            elif dp[i][j] == dp[i-1][j] + gap_penalty:
                aligned_a.append(sig_a[i-1])
                aligned_b.append(('─', 0))  # gap
                i -= 1
            else:
                aligned_a.append(('─', 0))  # gap
                aligned_b.append(sig_b[j-1])
                j -= 1
        elif i > 0:
            aligned_a.append(sig_a[i-1])
            aligned_b.append(('─', 0))
            i -= 1
        else:
            aligned_a.append(('─', 0))
            aligned_b.append(sig_b[j-1])
            j -= 1

    aligned_a.reverse()
    aligned_b.reverse()

    # final alignment score (normalized by length)
    final_score = dp[n][m] / max(n, m)

    return final_score, aligned_a, aligned_b

# build matrix_df here since we need it for signatures
matrix_df = pd.DataFrame(
    strength_matrix,
    index=valid_regions,
    columns=valid_regions
)

print("Connectivity signatures built.")
print(f"Each region has a signature of length {len(valid_regions)-1}")
print(f"\nSample signature for CA1:")
sig_ca1 = get_connectivity_signature('CA1', valid_regions, matrix_df)
for target, strength in sig_ca1[:5]:
    print(f"  CA1 → {target}: {strength:.4f}")
print("  ...")

Connectivity signatures built.
Each region has a signature of length 19

Sample signature for CA1:
  CA1 → ACAd: 0.1150
  CA1 → ACAv: 0.1177
  CA1 → CP: 0.1718
  CA1 → DG: 0.5430
  CA1 → ENTl: 0.1927
  ...


## Improves Sensitivity Alignment (pairwise)
Refined alignment using exponential match scoring and self-alignment normalization. Exponential scoring amplifies differences between connection strengths by spreading scores across the 0-1 range.  Biological validation: functionally related pairs (ACAd/ACAv:0.979) correctly score higher than unrelated pairs (CA1/VISp:0.808), confirming the algorith captures real wiring similarity.

In [10]:
def needleman_wunsch_sensitive(sig_a, sig_b, gap_penalty=-0.5):
    '''
    same algorithm but with:
    1. higher gap penalty (penalizes mismatches more)
    2. exponential match scoring (amplifies differences between strengths)
    this spreads scores out so similar vs different regions
    are more clearly separated
    '''
    n = len(sig_a)
    m = len(sig_b)

    dp = np.zeros((n+1, m+1))
    for i in range(n+1):
        dp[i][0] = i * gap_penalty
    for j in range(m+1):
        dp[0][j] = j * gap_penalty

    for i in range(1, n+1):
        for j in range(1, m+1):
            strength_diff = abs(sig_a[i-1][1] - sig_b[j-1][1])
            # exponential penalty: small differences are fine,
            # large differences are heavily penalized
            match_score = np.exp(-3 * strength_diff)

            diagonal = dp[i-1][j-1] + match_score
            up        = dp[i-1][j]   + gap_penalty
            left      = dp[i][j-1]   + gap_penalty
            dp[i][j]  = max(diagonal, up, left)

    # normalize by perfect self-alignment score
    # (what score would we get if we aligned sig_a with itself?)
    perfect_score = sum(np.exp(-3 * 0) for _ in sig_a)  # diff=0 for every position
    final_score = dp[n][m] / perfect_score

    return final_score

pairs_to_compare = [
    ('CA1',  'DG',    "both hippocampal subfields"),
    ('ACAd', 'ACAv',  "dorsal vs ventral anterior cingulate"),
    ('ENTl', 'ENTm',  "lateral vs medial entorhinal cortex"),
    ('VISp', 'VISl',  "primary vs lateral visual cortex"),
    ('CA1',  'VISp',  "hippocampus vs visual cortex (expect low)"),
    ('MOs',  'MOp',   "secondary vs primary motor cortex"),
]

results = []
print("CONNECTIVITY SIGNATURE ALIGNMENT (SENSITIVE VERSION)")
print("=" * 60)
print(f"{'Pair':<22} {'Score':>8}  {'Interpretation'}")
print("-" * 60)

for region_a, region_b, description in pairs_to_compare:
    sig_a = get_connectivity_signature(region_a, valid_regions, matrix_df)
    sig_b = get_connectivity_signature(region_b, valid_regions, matrix_df)
    score = needleman_wunsch_sensitive(sig_a, sig_b)
    results.append((region_a, region_b, score, description))

    if score > 0.85:
        interpretation = "very similar wiring"
    elif score > 0.70:
        interpretation = "moderately similar"
    elif score > 0.50:
        interpretation = "somewhat similar"
    else:
        interpretation = "different wiring patterns"

    print(f"{region_a+' vs '+region_b:<22} {score:>8.4f}  {interpretation}")

print()
best  = max(results, key=lambda x: x[2])
worst = min(results, key=lambda x: x[2])
print(f"Most similar:  {best[0]} vs {best[1]}: {best[2]:.4f}")
print(f"Most different: {worst[0]} vs {worst[1]}: {worst[2]:.4f}")
print(f"Spread: {best[2] - worst[2]:.4f}")

CONNECTIVITY SIGNATURE ALIGNMENT (SENSITIVE VERSION)
Pair                      Score  Interpretation
------------------------------------------------------------
CA1 vs DG                0.8620  very similar wiring
ACAd vs ACAv             0.9794  very similar wiring
ENTl vs ENTm             0.9693  very similar wiring
VISp vs VISl             0.9527  very similar wiring
CA1 vs VISp              0.8077  moderately similar
MOs vs MOp               0.9074  very similar wiring

Most similar:  ACAd vs ACAv: 0.9794
Most different: CA1 vs VISp: 0.8077
Spread: 0.1717


## Save Alignment Results to DB
Save improved alignment scores to the DB. DELETE FROM clears previous results while preserving the table schema. Results stored with biological descriptions for interpretability.

In [11]:
conn = sqlite3.connect('../db/neuro_nav.db')
cursor = conn.cursor()

# clear old results and save new ones
cursor.execute('DELETE FROM alignment_results')

for region_a, region_b, score, description in results:
    cursor.execute('''
        INSERT INTO alignment_results
        (region_a, region_b, alignment_score, description)
        VALUES (?, ?, ?, ?)
    ''', (region_a, region_b, round(score, 4), description))

conn.commit()
cursor.execute('SELECT * FROM alignment_results ORDER BY alignment_score DESC')
print("Updated alignment results:")
for row in cursor.fetchall():
    print(f"  {row[1]} vs {row[2]}: {row[3]:.4f} — {row[4]}")

conn.close()

Updated alignment results:
  ACAd vs ACAv: 0.9794 — dorsal vs ventral anterior cingulate
  ENTl vs ENTm: 0.9693 — lateral vs medial entorhinal cortex
  VISp vs VISl: 0.9527 — primary vs lateral visual cortex
  MOs vs MOp: 0.9074 — secondary vs primary motor cortex
  CA1 vs DG: 0.8620 — both hippocampal subfields
  CA1 vs VISp: 0.8077 — hippocampus vs visual cortex (expect low)
